# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankita0531/ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use **Logistic Regression** for this binary classification task.

The target is whether a content item experiences an impressions decline of more than 20% month-over-month. Logistic Regression fits this because it is a simple classification method that produces an interpretable probability and feature coefficients. This makes it useful for understanding which pre-outcome signals are associated with decline, rather than choosing a more complex model only because it may produce a higher score.

I will use the model as decision support and compare it fairly with the Week-4 hand-written baseline using the same held-out data and the same evaluation metric.


In [18]:
# Section 1: setup

%pip -q install duckdb huggingface_hub pandas scikit-learn

import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

con = duckdb.connect()

print("Libraries loaded and DuckDB connected.")

Libraries loaded and DuckDB connected.


In [19]:
# Section 2: split design

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

## 2. Split design

I will use a **grouped-by-client split** so that content from the same client does not appear in both training and test sets.

The model will use features available before the prediction window, while the label is calculated from the later outcome window. This prevents future information from entering the features.

A client-level split is appropriate because the model should generalize to unseen clients rather than only memorizing patterns from clients it has already seen. I will use 75% of the clients for training and 25% for testing, with a fixed random seed for reproducibility.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: split design

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", HF_TOKEN is not None)

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

# March 2026 is the prediction/outcome month.
# February 2026 provides the pre-outcome feature window.
PREV_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"
OUTCOME_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Feature window: February 2026")
print("Outcome window: March 2026")

HF token loaded: True
Feature window: February 2026
Outcome window: March 2026


In [21]:
# Section 2: build pre-outcome features + March outcome label

features = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM {PREV_REL}
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {OUTCOME_REL}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,
    feb.gsc_impressions,
    feb.gsc_clicks,
    feb.gsc_avg_position,
    feb.ga4_sessions,
    feb.ga4_engaged_sessions,
    mar.march_impressions,

    CASE
        WHEN mar.march_impressions < 0.8 * feb.gsc_impressions
        THEN 1
        ELSE 0
    END AS is_declining

FROM feb
INNER JOIN mar
    ON feb.client_hash_id = mar.client_hash_id
   AND feb.content_hash_id = mar.content_hash_id

WHERE feb.gsc_impressions > 0
""").df()

print("Modeling rows:", len(features))
print("\nTarget distribution:")
print(features["is_declining"].value_counts())
print("\nTarget rate:", features["is_declining"].mean())

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 21863

Target distribution:
is_declining
0    21473
1      390
Name: count, dtype: int64

Target rate: 0.017838357041577095


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_impressions,is_declining
0,client_65de48885f4ef01b,content_b08d07bc40bd93c8,12.0,0.0,2.333333,1.0,0.0,75.0,0
1,client_65de48885f4ef01b,content_806b0b68a12a5566,4.0,0.0,4.500000,1.0,0.0,28.0,0
2,client_65de48885f4ef01b,content_8764c5b599c52b68,17.0,0.0,6.529412,1.0,0.0,230.0,0
3,client_65de48885f4ef01b,content_47dbd73e4a9139db,56.0,0.0,3.696429,2.0,0.0,706.0,0
4,client_65de48885f4ef01b,content_74df5e33f1ad9383,9.0,1.0,9.333333,2.0,0.0,26.0,0


In [22]:
# Grouped split: clients cannot appear in both train and test

from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

model_data = features.dropna(subset=feature_cols + ["is_declining"]).copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_data,
        model_data["is_declining"],
        groups=model_data["client_hash_id"]
    )
)

train = model_data.iloc[train_idx].copy()
test = model_data.iloc[test_idx].copy()

X_train = train[feature_cols]
y_train = train["is_declining"]

X_test = test[feature_cols]
y_test = test["is_declining"]

print("Training rows:", len(train))
print("Test rows:", len(test))

print("\nTraining clients:", train["client_hash_id"].nunique())
print("Test clients:", test["client_hash_id"].nunique())

print("\nTrain target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Training rows: 20792
Test rows: 1071

Training clients: 14
Test clients: 5

Train target rate: 0.016592920353982302
Test target rate: 0.04201680672268908


## 3. Train + compare vs my baseline

### Logistic Regression

I train Logistic Regression using only the February pre-outcome features. The model predicts the probability that March impressions decline by more than 20%.

I will compare the model with the Week-4 hand-written baseline on the exact same held-out test clients. The comparison will use the same target and evaluation metrics so that model complexity is not rewarded by itself.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: train Logistic Regression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("Logistic Regression trained.")
print("\nClassification report:")
print(classification_report(
    y_test,
    test_pred,
    digits=3,
    zero_division=0
))

Logistic Regression trained.

Classification report:
              precision    recall  f1-score   support

           0      0.958     0.962     0.960      1026
           1      0.049     0.044     0.047        45

    accuracy                          0.923      1071
   macro avg      0.504     0.503     0.503      1071
weighted avg      0.920     0.923     0.922      1071



In [24]:
# Section 3: evaluate the Week-4 baseline on the same test set

baseline_test = test.copy()

baseline_test["baseline_score"] = np.select(
    [
        (baseline_test["gsc_impressions"] >= 100) &
        (baseline_test["gsc_avg_position"] > 10),

        (baseline_test["gsc_impressions"] >= 10) &
        (baseline_test["gsc_avg_position"] > 10)
    ],
    [2, 1],
    default=0
)

# Week-4 baseline predicts "priority/review" when score > 0
baseline_test["baseline_pred"] = (
    baseline_test["baseline_score"] > 0
).astype(int)

print("Week-4 baseline classification report:")
print(classification_report(
    y_test,
    baseline_test["baseline_pred"],
    digits=3,
    zero_division=0
))

Week-4 baseline classification report:
              precision    recall  f1-score   support

           0      0.962     0.868     0.913      1026
           1      0.069     0.222     0.105        45

    accuracy                          0.841      1071
   macro avg      0.516     0.545     0.509      1071
weighted avg      0.925     0.841     0.879      1071



In [25]:
# Compare Logistic Regression vs Week-4 baseline

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "accuracy": [
        accuracy_score(y_test, baseline_test["baseline_pred"]),
        accuracy_score(y_test, test_pred)
    ],
    "precision": [
        precision_score(y_test, baseline_test["baseline_pred"], zero_division=0),
        precision_score(y_test, test_pred, zero_division=0)
    ],
    "recall": [
        recall_score(y_test, baseline_test["baseline_pred"], zero_division=0),
        recall_score(y_test, test_pred, zero_division=0)
    ],
    "f1": [
        f1_score(y_test, baseline_test["baseline_pred"], zero_division=0),
        f1_score(y_test, test_pred, zero_division=0)
    ]
})

comparison

,method,accuracy,precision,recall,f1
0,Week-4 baseline,0.841270,0.068966,0.222222,0.105263
1,Logistic Regression,0.923436,0.048780,0.044444,0.046512


### Model vs baseline interpretation

The Logistic Regression achieves higher overall accuracy (0.923 vs 0.841), but accuracy is not the most informative metric because the declining class is rare.

The Week-4 baseline performs better on the positive class: precision is 0.069, recall is 0.222, and F1 is 0.105, compared with 0.049 precision, 0.044 recall, and 0.047 F1 for Logistic Regression.

Therefore, Logistic Regression does not improve the baseline on the positive-class metrics at the current 0.5 decision threshold. The baseline is better at identifying declining content, while the model produces fewer positive predictions. This shows that model complexity alone does not guarantee a better action-ranking rule.

In [26]:
# Interpret Logistic Regression coefficients

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.named_steps["logistic"].coef_[0]
})

coef_table["abs_coefficient"] = coef_table["coefficient"].abs()

coef_table.sort_values(
    "abs_coefficient",
    ascending=False
)

,feature,coefficient,abs_coefficient
1,gsc_clicks,0.773871,0.773871
2,gsc_avg_position,-0.617675,0.617675
4,ga4_engaged_sessions,0.500660,0.500660
3,ga4_sessions,-0.382858,0.382858
0,gsc_impressions,0.374685,0.374685


## 4. Errors and interpretation

### Feature interpretation

The Logistic Regression coefficients show which standardized features are most associated with the predicted decline class.

`gsc_clicks` has the largest positive coefficient (0.774), followed by `ga4_engaged_sessions` (0.501) and `gsc_impressions` (0.375). `gsc_avg_position` has a negative coefficient (-0.618), while `ga4_sessions` also has a negative coefficient (-0.383).

These coefficients are associations rather than causal effects. The model is useful for interpreting the available signals, but the weak positive-class F1 shows that these features alone do not identify declining content particularly well on the held-out clients.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect false positives and false negatives

error_analysis = test.copy()

error_analysis["predicted"] = test_pred
error_analysis["predicted_probability"] = test_prob

error_analysis["error_type"] = np.select(
    [
        (error_analysis["is_declining"] == 1) &
        (error_analysis["predicted"] == 0),

        (error_analysis["is_declining"] == 0) &
        (error_analysis["predicted"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nFalse negatives:")
display(
    error_analysis[
        error_analysis["error_type"] == "false_negative"
    ][
        feature_cols +
        ["march_impressions", "is_declining", "predicted_probability"]
    ].head(10)
)

print("\nFalse positives:")
display(
    error_analysis[
        error_analysis["error_type"] == "false_positive"
    ][
        feature_cols +
        ["march_impressions", "is_declining", "predicted_probability"]
    ].head(10)
)

Error counts:
error_type
correct           989
false_negative     43
false_positive     39
Name: count, dtype: int64

False negatives:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_impressions,is_declining,predicted_probability
1907,7.0,1.0,8.142857,1.0,0.0,3.0,1,0.390275
2137,143.0,2.0,8.272727,4.0,0.0,69.0,1,0.392253
2148,16.0,0.0,37.812500,3.0,0.0,7.0,1,0.123365
8457,4.0,0.0,16.750000,3.0,0.0,3.0,1,0.285471
8465,5.0,0.0,37.800000,1.0,0.0,1.0,1,0.125137
9793,14.0,0.0,11.071429,3.0,0.0,10.0,1,0.346384
10628,25.0,0.0,7.480000,1.0,0.0,17.0,1,0.391924
10803,4.0,0.0,86.250000,1.0,0.0,2.0,1,0.012774
10805,5.0,0.0,16.600000,1.0,0.0,2.0,1,0.290429
10830,64.0,2.0,6.000000,2.0,0.0,37.0,1,0.421759



False positives:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_impressions,is_declining,predicted_probability
1886,8572.0,34.0,7.167056,47.0,1.0,17300.0,0,0.739946
1893,21566.0,29.0,6.444913,51.0,1.0,37969.0,0,0.888166
1894,4708.0,12.0,7.965803,16.0,0.0,6388.0,0,0.540214
1906,8884.0,2.0,6.866389,8.0,0.0,38813.0,0,0.593895
1977,1037.0,11.0,4.405014,13.0,1.0,2842.0,0,0.539461
1997,9899.0,20.0,6.183756,20.0,3.0,28846.0,0,0.794847
2015,3205.0,17.0,5.811232,19.0,1.0,7814.0,0,0.598129
2016,1485.0,6.0,5.468013,13.0,2.0,6587.0,0,0.539638
2042,1567.0,8.0,2.491385,9.0,0.0,3806.0,0,0.524955
2046,456.0,4.0,0.839912,4.0,1.0,787.0,0,0.541166


### Error analysis

The test set contains 43 false negatives and 39 false positives.

The false negatives show that some content experienced a real March decline even though the model assigned a probability below the 0.5 classification threshold. Several of these cases had relatively low February traffic and clicks, suggesting that low-volume content can be difficult for the model to identify as a future decline.

The false positives show the opposite problem: some content received a high predicted decline probability but did not actually decline. Many of these examples had strong February search activity and engagement, so the model appears to associate some high-activity patterns with decline even when the later outcome does not confirm it.

Overall, the errors suggest that the five available pre-outcome features do not fully explain March declines. Additional temporal or content-level signals may be needed to improve the model.

In [28]:
# Positive-class rate and prediction coverage

print("Actual decline rate:", round(y_test.mean(), 4))
print("Model predicted decline rate:", round(test_pred.mean(), 4))
print("Baseline predicted decline rate:",
      round(baseline_test["baseline_pred"].mean(), 4))

Actual decline rate: 0.042
Model predicted decline rate: 0.0383
Baseline predicted decline rate: 0.1354


### Overall interpretation

The test set has a 4.2% actual decline rate. Logistic Regression predicts 3.83% of test rows as declining, which is close to the observed rate. The Week-4 baseline flags 13.54% of rows, making it substantially more aggressive.

This difference explains the trade-off between the two approaches. The baseline identifies more of the declining cases, giving it higher recall and F1, while Logistic Regression makes fewer positive predictions and therefore misses more actual declines.

For this experiment, the Week-4 baseline remains the stronger decision rule for finding declining content. Logistic Regression provides useful feature interpretation, but its current classification threshold and feature set do not produce a better positive-class result.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.